# Fix Q3 2025's campaign keys

*Data Analysis in Python · Topic 5.3*

The video team says the paid media report gives YouTube no revenue at all for July to September 2025. Find the spend that doesn't match the store's revenue, measure what it costs, and fix the campaign keys.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps), the cell that builds the `revenue` table (topic 5.2.'s steps) and the cell that builds the `spend` table: the three platforms' spend for July to September 2025, in topic 5.1.'s standard columns.
2. In the first cell marked ✏️, create `unmatched_names`: the set of campaign names in the `spend` table that the `revenue` table doesn't have. Look at them with `sorted()`.
3. In the second cell marked ✏️, create the `check` table: the `spend` table joined to the `revenue` table on the date and the campaign, keeping every row of the `spend` table, with a `_merge` column. Then create:
   - `unmatched_rows`: the number of rows of the `check` table that found no revenue
   - `spend_lost`: their total spend, rounded to 2 decimals
4. In the last cell marked ✏️, write `campaign_key(names)`, which returns a column of campaign names stripped of spaces and lowercased. Use it on the `campaign` column of both the `spend` and the `revenue` table. Then create:
   - `unmatched_after`: the same set as `unmatched_names`, after the fix
   - `youtube_roas`: YouTube's revenue over its spend, from the `spend` table joined to the `revenue` table again
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `unmatched_names` is a set of four names: two YouTube campaigns with a capital market, and two Meta campaigns with a trailing space. `unmatched_rows` is `474` and `spend_lost` is `13965.82`. After the fix, `unmatched_after` is an empty set, and `youtube_roas` is about `1.4766`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-07", "2025-08", "2025-09"]
START, END = "2025-07-01", "2025-09-30"
COLUMNS = ["date", "platform", "campaign", "ad_set", "impressions", "clicks", "spend"]

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Revenue per date and campaign, with topic 5.2's steps. Run it, and leave it as it is.
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue.shape

In [ ]:
# The three platforms' spend, with topic 5.1's steps. Run it, and leave it as it is.
def standard_columns(table):
    """Return the table with only the standard columns it has, in the standard order."""
    return table[[column for column in COLUMNS if column in table.columns]]


google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
google_std = google.rename(columns={"campaign_name": "campaign"})
google_std["platform"] = "Google Ads"
google_std["spend"] = google_std["cost_micros"] / 1_000_000

meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
meta_std = meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Ad set name": "ad_set",
                                "Impressions": "impressions", "Link clicks": "clicks", "Amount spent (USD)": "spend"})
meta_std["platform"] = "Meta"

youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
youtube_std = youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Impr.": "impressions", "Clicks": "clicks", "Cost": "spend"})
youtube_std["platform"] = "YouTube"

spend = pd.concat([standard_columns(table) for table in [google_std, meta_std, youtube_std]], ignore_index=True)[COLUMNS]
spend = spend[spend["date"].between(START, END)].reset_index(drop=True)
spend.groupby("platform")["spend"].sum()

## The keys

In [ ]:
# ✏️ Create unmatched_names


In [ ]:
# ✏️ Create check, unmatched_rows and spend_lost


## The fix

In [ ]:
# ✏️ Write campaign_key(), fix both tables, and create unmatched_after and youtube_roas


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    expected_names = {"2025_CA_youtube_awareness", "2025_US_youtube_awareness", "2025_ca_facebook_prospecting ", "2025_ca_facebook_summersale "}
    names = found.get("unmatched_names")
    if isinstance(names, (list, tuple, pd.Series)) and set(names) == expected_names:
        results.append((False, "unmatched_names has the right four names, but it should be a set: set(spend[\"campaign\"]) - set(revenue[\"campaign\"])."))
    elif not isinstance(names, set):
        results.append((False, "unmatched_names isn't a set yet. Subtract one set of names from the other: set(spend[\"campaign\"]) - set(revenue[\"campaign\"])."))
    elif "(direct)" in names or "(organic)" in names:
        results.append((False, f"unmatched_names has {len(names)} names, such as (direct) and (organic): the revenue table's names that the spend table doesn't have. Subtract the other way round: spend's names minus revenue's."))
    elif names == set():
        results.append((False, "unmatched_names is empty: it was made after the keys were fixed, or from fixed names. Make it from the spend and revenue tables as they are, before the fix."))
    elif names != expected_names:
        results.append((False, f"unmatched_names should be spend's campaign names that revenue doesn't have, before the fix: four of them. It is {sorted(names)}."))
    else:
        results.append((True, "Four names don't match: two YouTube campaigns with a capital market, and two Meta campaigns that end in a space."))

    check = found.get("check")
    rows = found.get("unmatched_rows")
    lost = found.get("spend_lost")
    if not isinstance(check, pd.DataFrame) or "_merge" not in check.columns:
        results.append((False, "check should be the spend table joined to the revenue table with a _merge column: spend.merge(revenue, on=[\"date\", \"campaign\"], how=\"left\", indicator=True)."))
    elif len(check) != 2646:
        results.append((False, f"The check table has {len(check):,} rows, but the spend table has 2,646. Keep every row of the spend table: how=\"left\"."))
    elif (check["_merge"] == "left_only").sum() == 0:
        results.append((False, "The check table has no left_only rows: it was joined after the keys were fixed. Join the tables as they are first, to see what the mismatch costs."))
    elif isinstance(rows, (pd.DataFrame, pd.Series)):
        results.append((False, f"unmatched_rows is a {type(rows).__name__}, not a number. Count the rows: (check[\"_merge\"] == \"left_only\").sum()."))
    elif not pd.api.types.is_number(rows) or rows != 474:
        results.append((False, f"unmatched_rows should be the number of left_only rows in the check table, 474. It is {rows!r}."))
    elif isinstance(lost, (pd.DataFrame, pd.Series)):
        results.append((False, f"spend_lost is a {type(lost).__name__}, not a number. Total the left_only rows' spend column, and round it."))
    elif pd.api.types.is_number(lost) and abs(lost - 12102.60) < 0.005:
        results.append((False, "spend_lost is 12,102.60: YouTube's only. Meta's two names with a trailing space lose another $1,863.22."))
    elif not pd.api.types.is_number(lost) or abs(lost - 13965.82) > 0.005:
        results.append((False, f"spend_lost should be the left_only rows' spend, rounded to 2 decimals, 13965.82. It is {lost!r}."))
    else:
        results.append((True, "474 rows of spend found no revenue: $13,965.82, YouTube's whole quarter and some of Meta's."))

    spend = found.get("spend")
    revenue = found.get("revenue")
    campaign_key = found.get("campaign_key")
    after = found.get("unmatched_after")
    youtube_roas = found.get("youtube_roas")
    if not callable(campaign_key):
        results.append((False, "campaign_key isn't a function yet. def campaign_key(names): return names.str.strip().str.lower()."))
    elif list(campaign_key(pd.Series([" 2025_US_youtube_awareness ", "2025_ca_facebook_prospecting "]))) != ["2025_us_youtube_awareness", "2025_ca_facebook_prospecting"]:
        results.append((False, f"campaign_key() should strip spaces and lowercase. For \" 2025_US_youtube_awareness \" it gives {campaign_key(pd.Series([' 2025_US_youtube_awareness ']))[0]!r}."))
    elif not isinstance(spend, pd.DataFrame) or not isinstance(revenue, pd.DataFrame) or (spend["campaign"] != campaign_key(spend["campaign"])).any():
        results.append((False, "The spend table's campaign names aren't fixed yet. spend[\"campaign\"] = campaign_key(spend[\"campaign\"]), and the same for revenue."))
    elif history is not None and not any("campaign_key(revenue" in source.replace(" ", "") or "revenue[\"campaign\"]=campaign_key" in source.replace(" ", "") for source in history):
        results.append((False, "The spend table's names are fixed, but nothing in your notebook runs campaign_key() on the revenue table's. Fix both sides with the same function, so the two can't drift apart."))
    elif not isinstance(after, set):
        results.append((False, "unmatched_after isn't a set yet. Subtract the sets again, after the fix."))
    elif after:
        results.append((False, f"unmatched_after still has {sorted(after)}. Strip and lowercase the names."))
    elif isinstance(youtube_roas, (pd.DataFrame, pd.Series)):
        results.append((False, f"youtube_roas is a {type(youtube_roas).__name__}, not a number. Keep YouTube's rows of the joined table, and divide its revenue total by its spend total."))
    elif not pd.api.types.is_number(youtube_roas):
        results.append((False, f"youtube_roas should be YouTube's revenue over its spend, after the fix. It is {youtube_roas!r}."))
    elif abs(youtube_roas) < 0.0001:
        results.append((False, "youtube_roas is 0: the join ran before the fix, so YouTube's rows found no revenue. Join again after fixing both tables."))
    elif abs(youtube_roas - 1.4766) > 0.0001:
        results.append((False, f"youtube_roas should be the YouTube rows' revenue total over their spend total, about 1.4766. It is {youtube_roas!r}."))
    else:
        results.append((True, f"Every campaign matches after the fix, and YouTube's ROAS is {youtube_roas:.2f}, not zero."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
unmatched_names = set(spend["campaign"]) - set(revenue["campaign"])
sorted(unmatched_names)
```

```python
check = spend.merge(revenue, on=["date", "campaign"], how="left", indicator=True)
unmatched_rows = (check["_merge"] == "left_only").sum()
spend_lost = round(check.loc[check["_merge"] == "left_only", "spend"].sum(), 2)
check["_merge"].value_counts(), unmatched_rows, spend_lost
```

```python
def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


spend["campaign"] = campaign_key(spend["campaign"])
revenue["campaign"] = campaign_key(revenue["campaign"])
unmatched_after = set(spend["campaign"]) - set(revenue["campaign"])

joined = spend.merge(revenue, on=["date", "campaign"], how="left")
youtube_rows = joined[joined["platform"] == "YouTube"]
youtube_roas = youtube_rows["revenue"].sum() / youtube_rows["spend"].sum()
unmatched_after, youtube_roas
```

</details>